# 03c. Does the choice of gradient-boosting library matter here?

*Rent Reality Check (London). Created by Kevin Steepan.*

---

## Why this notebook

Notebook 03 uses scikit-learn's `HistGradientBoostingRegressor`. There are three
libraries people usually reach for on tabular data: **LightGBM**, **XGBoost**, and
scikit-learn's histogram booster (which is LightGBM's idea, ported into
scikit-learn). A fair question from anyone reading this is: would one of the others
do noticeably better?

This is a small check, not a competition. Each library gets its **default**
settings and the **same spatial folds** from notebook 03. No tuning: the point is
to see whether the library choice alone moves the number.

> **Where this comes from.** LightGBM: Ke et al. (2017). XGBoost: Chen and
> Guestrin (2016). scikit-learn's `HistGradientBoosting*` follows the LightGBM
> histogram approach. All three are covered in the scikit-learn ensemble user
> guide and in most applied-ML courses as "the tabular default".

In [1]:
import time
import numpy as np
import pandas as pd

from londonrent.features import build_model_frame
from londonrent import model as M

# cold-start feature set, the one the app ships
frame, groups = build_model_frame(with_reviews=False)
y = frame["log_price"].to_numpy()
enc = M.fit_encoder(frame, groups["categorical"])
X = M.build_design_matrix(frame, groups, enc)
folds = M.spatial_block_folds(frame["latitude"].to_numpy(),
                              frame["longitude"].to_numpy(), n_splits=5, seed=0)
print(f"{len(frame):,} listings, design matrix {X.shape}")

59,450 listings, design matrix (59450, 72)


In [2]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

def oof_score(make):
    '''Out-of-fold predictions with the spatial folds, then notebook 03's metrics.'''
    pred = np.full(len(y), np.nan)
    t0 = time.time()
    for tr, te in folds:
        m = make()
        m.fit(X.iloc[tr], y[tr])
        pred[te] = m.predict(X.iloc[te])
    out = M.evaluate(y, pred)
    out["fit_seconds"] = round(time.time() - t0, 1)
    return out

candidates = {
    "sklearn HistGradientBoosting (default)": lambda: M.make_model(),
    "LightGBM (default)":  lambda: LGBMRegressor(random_state=0, verbose=-1),
    "XGBoost (default)":   lambda: XGBRegressor(random_state=0, verbosity=0),
}
results = pd.DataFrame({name: oof_score(mk) for name, mk in candidates.items()}).T
results.round(4)[["MAE_gbp", "MdAPE", "within_15pct", "R2_log", "fit_seconds"]]

,MAE_gbp,MdAPE,within_15pct,R2_log,fit_seconds
sklearn HistGradientBoosting (default),61.8509,0.2228,0.3508,0.7177,72.0
LightGBM (default),61.7671,0.2214,0.3535,0.7181,3.5
XGBoost (default),63.3164,0.2267,0.3474,0.7056,3.4


**What we see.**

| model | £-MAE | median % err | R-squared | fit time |
|---|---|---|---|---|
| sklearn HistGradientBoosting (default) | 61.85 | 22.3% | 0.718 | 72 s |
| LightGBM (default) | 61.77 | 22.1% | 0.718 | 3.5 s |
| XGBoost (default) | 63.32 | 22.7% | 0.706 | 3.4 s |

- **LightGBM and the scikit-learn booster are the same model to two decimal
  places.** No surprise: the scikit-learn one is the LightGBM histogram method
  reimplemented.
- **XGBoost's defaults are a little behind** (about £1.50 worse on MAE, 0.01 lower
  R-squared). Its out-of-the-box settings are less tuned for this shape of
  problem; a proper tune would close most of that gap.
- **The 72-second fit time is my configuration, not the library.** Notebook 03's
  `DEFAULT_PARAMS` sets `max_iter=600` with early stopping; LightGBM and XGBoost
  default to 100 trees. Drop the scikit-learn one to `max_iter=100` and it fits in
  a few seconds too.

**Why notebook 03 uses the scikit-learn one:**

- No system-library dependency. LightGBM and XGBoost both need OpenMP (`libomp`)
  present, which is one more thing that can break a deploy. The Streamlit Cloud
  build stays smaller and simpler without them.
- It handles missing values natively, so no imputation step.
- It is one `import` from a library already in the project.

If this were a Kaggle competition rather than a portfolio piece, I would tune
LightGBM properly and use that. For a model that has to load in a small hosted
app, the scikit-learn one is the sensible default.